# Lab 4: Data pipeline and quality

Reproducible UCI Adult experiment. Run this notebook from its project folder
using Python 3.11 with the pinned requirements installed. The notebook calls
the same implementation as `python run.py`; it does not maintain a second copy.

**Manual 50-row review is pending** in `results/audit_50_manual.csv`.


In [ ]:
from pathlib import Path
import json
import yaml
import pandas as pd
from IPython.display import display, Markdown, Image
from run import ROOT, main


## 1. Problem definition and configuration

Predict >50K income from 14 census attributes. Positive label = >50K.
Report accuracy and binary F1. Use a stratified 70/30 split with seed 42,
without learning preprocessing parameters from test. The half-page problem
definition and final discussion are in REPORT.md.


In [ ]:
cfg = yaml.safe_load((ROOT / 'config.yaml').read_text())
cfg


## 2-7. Execute the complete pipeline

The implementation ingests raw data without modification, validates all 15
columns, splits before cleaning, drops exact records from training only,
fits train medians/modes and 1st/99th-percentile clipping, then fits train
scaling and one-hot encoding. It evaluates LogisticRegression with and
without capital_net and hours_per_age, performs the conflict audit and
writes hashes, saved models and documentation.

Data validation failures stop execution after saving the report. The existing
human review worksheet is preserved on reruns.


In [ ]:
manifest = main()


## Validation: dimensions, missingness and reported violations


In [ ]:
validation = json.loads((ROOT / 'results/validation_report.json').read_text())
print('Shape:', validation['shape'], 'Valid:', validation['valid'])
display(pd.DataFrame(validation['columns']).T[['expected_type', 'actual_dtype', 'missing_total']])
print('Violations:', validation['violations'])


## Train/test integrity

No test rows are removed. Identical feature vectors can cross the prescribed row split; the report discloses the overlap.


In [ ]:
{k: manifest[k] for k in ['train_rows_before_dedup', 'train_duplicates_removed', 'train_rows', 'test_rows', 'test_rows_with_feature_match_in_train', 'automatic_checks']}


## Feature engineering comparison

Derived features are calculated after imputation and clipping; both variants use identical split rows and fixed model settings.


In [ ]:
metrics = json.loads((ROOT / 'results/metrics.json').read_text())
display(pd.DataFrame(metrics).T[['accuracy', 'f1', 'encoded_features']])
display(Image(filename=str(ROOT / 'results/metrics.png')))


## Label audit and 50 sampled rows

Automated dictionary checks are supplied. Inspect the separate manual CSV yourself and complete its status and notes. Valid ranges do not verify actual income. No 24-hour relabelling or Cohen's kappa is required for the Adult branch.


In [ ]:
display(json.loads((ROOT / 'results/label_quality_report.json').read_text()))
display(pd.read_csv(ROOT / 'results/audit_50_manual.csv', keep_default_na=False))


## Processed-data SHA256 and independent checks


In [ ]:
print((ROOT / 'results/processed_dataset.sha256').read_text())
from verify import main as verify_results
verify_results()


## Final report and data card


In [ ]:
display(Markdown((ROOT / 'REPORT.md').read_text(encoding='utf-8')))
display(Markdown((ROOT / 'data_card.md').read_text(encoding='utf-8')))
